## Create QE score using an LLM-based approach

In [1]:
#%pip install openai python-dotenv
import pandas as pd
from dotenv import load_dotenv
import os
from openai import OpenAI

#%pip install git+https://github.com/MicrosoftTranslator/GEMBA.git
from gemba.gemba_mqm_utils import (
    TEMPLATE_GEMBA_MQM,
    apply_template,
    parse_mqm_answer
)

In [2]:
data_path = '/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/data/dataset_qe/divemt_cometkiwi_qescore.tsv'
df = pd.read_csv(data_path, sep ='\t')
#print(df.shape)
#print(df.columns)

In [3]:
#Create dictionary of language code mapped to its language name
languages = {
    'tur':'Turkish',
    'ita':'Italian',
    'ukr':'Ukranian',
    'nld':'Dutch',
    'vie':'Vietnamese',
    'ara':'Arabic'
}

In [4]:
load_dotenv()
api_key = os.getenv('OPENROUTER_API_KEY')
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

open_model = "google/gemma-3-27b-it"
proprietary_model = "openai/gpt-4.1-mini"


In [5]:
print (api_key is not None)

True


In [6]:
gemba_input_df = df.copy()
#add source language feature to df, value for all cells is English
gemba_input_df['source_lang'] = "English"
#add target language feature to df, value is taken from the languages dictionary
gemba_input_df['target_lang'] = gemba_input_df['lang_id'].map(languages)

print(gemba_input_df['source_lang'].shape)
print (gemba_input_df['source_lang'].unique())
print (gemba_input_df['source_lang'].nunique())
print(gemba_input_df.shape)
gemba_input_df.head(5)


(5160,)
['English']
1
(5160, 67)


,unit_id,flores_id,item_id,subject_id,task_type,translation_type,src_len_chr,mt_len_chr,tgt_len_chr,src_len_wrd,...,src_annotations,mt_tokens,mt_annotations,tgt_tokens,tgt_annotations,src_wmt22_qe,mt_wmt22_qe,qe_cometkiwi,source_lang,target_lang
0,flores101-main-tur-100-pe1-1,894,flores101-main-1001,tur_t1,pe1,pe,124,126.0,128,21,...,"[{'lemma': 're-entry', 'upos': 'NOUN', 'feats'...","['Yeniden', 'giriş', 'şoku', 'kültür', 'şokund...","[{'lemma': 'yeniden', 'upos': 'ADV', 'feats': ...","['Yeniden', 'giriş', 'şoku', ',', 'kültür', 'ş...","[{'lemma': 'yeniden', 'upos': 'ADV', 'feats': ...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...","['OK', 'OK', 'OK', 'BAD', 'OK', 'OK', 'OK', 'O...",0.863117,English,Turkish
1,flores101-main-tur-100-pe1-2,895,flores101-main-1002,tur_t1,pe1,pe,137,117.0,117,22,...,"[{'lemma': 'traveller', 'upos': 'NOUN', 'feats...","['Yeni', 'kültüre', 'kolayca', 'uyum', 'sağlay...","[{'lemma': 'yeni', 'upos': 'ADJ', 'feats': '',...","['Yeni', 'kültüre', 'kolayca', 'uyum', 'sağlay...","[{'lemma': 'yeni', 'upos': 'ADJ', 'feats': '',...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...",0.868266,English,Turkish
2,flores101-main-tur-100-pe1-3,896,flores101-main-1003,tur_t1,pe1,pe,127,133.0,148,22,...,"[{'lemma': 'when', 'upos': 'SCONJ', 'feats': '...","['Yurtdışında', 'yaşadıktan', 'sonra', 'eve', ...","[{'lemma': 'yurtdışı', 'upos': 'NOUN', 'feats'...","['Yurtdışında', 'yaşadıktan', 'sonra', 'ülkeni...","[{'lemma': 'yurtdışı', 'upos': 'NOUN', 'feats'...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'BAD', 'B...","['OK', 'OK', 'OK', 'BAD', 'OK', 'BAD', 'OK', '...",0.891622,English,Turkish
3,flores101-main-tur-100-pe1-4,897,flores101-main-1004,tur_t1,pe1,pe,134,145.0,166,22,...,"[{'lemma': 'when', 'upos': 'SCONJ', 'feats': '...","['İlk', 'başta', 'yurt', 'dışına', 'çıktığınız...","[{'lemma': 'ilk', 'upos': 'ADJ', 'feats': '', ...","['Yurt', 'dışına', 'ilk', 'çıktığınızda', 'çev...","[{'lemma': 'yurt', 'upos': 'NOUN', 'feats': 'C...","['OK', 'OK', 'OK', 'BAD', 'OK', 'BAD', 'BAD', ...","['BAD', 'BAD', 'BAD', 'OK', 'BAD', 'BAD', 'OK'...",0.865835,English,Turkish
4,flores101-main-tur-100-pe1-5,898,flores101-main-1005,tur_t1,pe1,pe,107,89.0,94,15,...,"[{'lemma': 'person', 'upos': 'NOUN', 'feats': ...","['İnsanlar', 'bu', 'sabır', 'tahmin', 'edemeye...","[{'lemma': 'insan', 'upos': 'NOUN', 'feats': '...","['İnsanlar', ',', 'ülkesine', 'dönenlere', 'de...","[{'lemma': 'insan', 'upos': 'NOUN', 'feats': '...","['OK', 'BAD', 'OK', 'BAD', 'BAD', 'BAD', 'OK',...","['OK', 'BAD', 'BAD', 'BAD', 'BAD', 'OK', 'OK',...",0.635008,English,Turkish


In [7]:
#Prepare inputs for GEMBA

gemba_inputs = []

for _, row in gemba_input_df.iterrows():
    segment_data ={
        "source_lang": row["source_lang"],
        "target_lang": row["target_lang"],
        "source_seg": row["src_text"],
        "target_seg": row["mt_text"]
        }
    
    input_item = {
        "unit_id": row["unit_id"],
        "segment_data": segment_data
    }
    gemba_inputs.append(input_item)


In [10]:
# Create qe score with open model google/gemma-3-27b-it

gemma_results=[] #list to store results from Gemma

for i, item in enumerate (gemba_inputs, start =1):

    messages = apply_template(
        TEMPLATE_GEMBA_MQM,
        item['segment_data']
    )

    try:
        response = client.chat.completions.create(
            model = open_model,
            messages = messages,
            temperature = 0
        )
        raw_output = response.choices[0].message.content
        
        score = parse_mqm_answer (raw_output)
        
        errors = parse_mqm_answer(
            raw_output,
            list_mqm_errors = True
        )
        gemma_results.append(
            {
                'unit_id': item['unit_id'],
                'qe_gemba_gemma3': score,
                'gemba_gemma3_errors': dict(errors),
                'gemba_gemma_raw_output': raw_output
            }
        )

    except Exception as e:
        gemma_results.append(
            {
                'unit_id': item['unit_id'],
                'qe_gemba_gemma3': None,
                'gemba_gemma3_errors': None,
                'gemba_gemma3_raw_output': None  
            }
        )
        print (f'Error at row {i}:{e}')

    if i % 100 == 0:
        checkpoint_df = pd.DataFrame (gemma_results)
        checkpoint_df.to_csv("../data/dataset_qe/gemba_gemma3_qe_outputs.tsv", sep ='\t', index = False)


the primary issue is the character encoding. the use of "q" instead of "k" in "qaranlıq" makes the word incorrect and difficult to understand for a turkish speaker. this is a *critical* error because it directly impacts readability and meaning. the rest of the translation appears grammatically correct and conveys the original meaning, though it might not be the most elegant turkish phrasing.
the primary issue is the character encoding. the use of "q" instead of "k" in "qaranlıq" makes the word incorrect and difficult to understand for a turkish speaker. this is a *critical* error because it directly impacts readability and meaning. the rest of the translation appears grammatically correct and conveys the original meaning, though it might not be the most elegant turkish phrasing.
the primary issue is the character encoding. "müxtəlifliği" appears to be azerbaijani or another similar language, not turkish. this renders the word unrecognizable and significantly impacts comprehension. whil

In [11]:
print(checkpoint_df.shape)
print(checkpoint_df.head())
print(checkpoint_df["qe_gemba_gemma3"].isna().sum())

(5100, 4)
                        unit_id  qe_gemba_gemma3  \
0  flores101-main-tur-100-pe1-1                0   
1  flores101-main-tur-100-pe1-2                0   
2  flores101-main-tur-100-pe1-3                0   
3  flores101-main-tur-100-pe1-4               -1   
4  flores101-main-tur-100-pe1-5              -25   

                                 gemba_gemma3_errors  \
0                                                 {}   
1                                                 {}   
2                                                 {}   
3  {'minor': ['fluency/grammar - the sentence str...   
4  {'critical': ['accuracy/mistranslation - the s...   

                              gemba_gemma_raw_output  
0  \nCritical:\nno-error\nMajor:\nno-error\nMinor...  
1  \nCritical:\nno-error\nMajor:\nno-error\nMinor...  
2  \nCritical:\nno-error\nMajor:\nno-error\nMinor...  
3  \nCritical:\nno-error\nMajor:\nno-error\nMinor...  
4  \nCritical:\naccuracy/mistranslation - The sen...  
0


In [17]:
gemma_results_df = pd.DataFrame(gemma_results)

"""
gemma_results_df.to_csv(
    "../data/dataset_qe/gemba_gemma3_qe_outputs.tsv",
    sep="\t",
    index=False
)
"""

print(gemma_results_df.shape)
print(gemma_results_df["qe_gemba_gemma3"].isna().sum())
gemma_results_df.head(10)

(5160, 4)
0


,unit_id,qe_gemba_gemma3,gemba_gemma3_errors,gemba_gemma_raw_output
0,flores101-main-tur-100-pe1-1,0,{},\nCritical:\nno-error\nMajor:\nno-error\nMinor...
1,flores101-main-tur-100-pe1-2,0,{},\nCritical:\nno-error\nMajor:\nno-error\nMinor...
2,flores101-main-tur-100-pe1-3,0,{},\nCritical:\nno-error\nMajor:\nno-error\nMinor...
3,flores101-main-tur-100-pe1-4,-1,{'minor': ['fluency/grammar - the sentence str...,\nCritical:\nno-error\nMajor:\nno-error\nMinor...
4,flores101-main-tur-100-pe1-5,-25,{'critical': ['accuracy/mistranslation - the s...,\nCritical:\naccuracy/mistranslation - The sen...
5,flores101-main-tur-101-pe2-1,-1,{'minor': ['fluency/punctuation - missing a fu...,\nCritical:\nno-error\nMajor:\nno-error\nMinor...
6,flores101-main-tur-101-pe2-2,0,{},\nCritical:\nno-error\nMajor:\nno-error\nMinor...
7,flores101-main-tur-101-pe2-3,-6,{'major': ['fluency/grammar - the sentence str...,\nCritical:\nno-error\n\nMajor:\nfluency/gramm...
8,flores101-main-tur-103-pe1-1,-1,"{'minor': ['style/awkward - ""ki"" (the use of ""...",\nCritical:\nno-error\nMajor:\nno-error\nMinor...
9,flores101-main-tur-103-pe1-2,-6,{'major': ['terminology/inappropriate for cont...,\nCritical:\nno-error\nMajor:\nterminology/ina...


In [18]:
# Create qe score with proprietary model openai/gpt-4.1-mini

gpt_results=[] #list to store results from Gpt4

for i, item in enumerate (gemba_inputs, start =1):

    messages = apply_template(
        TEMPLATE_GEMBA_MQM,
        item['segment_data']
    )

    try:
        response = client.chat.completions.create(
            model = proprietary_model,
            messages = messages,
            temperature = 0
        )
        raw_output = response.choices[0].message.content
        
        score = parse_mqm_answer (raw_output)
        
        errors = parse_mqm_answer(
            raw_output,
            list_mqm_errors = True
        )
        gpt_results.append(
            {
                'unit_id': item['unit_id'],
                'qe_gemba_gpt4_1_mini': score,
                'gemba_gpt4_1_mini_errors': dict(errors),
                'gemba_gpt4_1_mini_raw_output': raw_output
            }
        )

    except Exception as e:
        gpt_results.append(
            {
                'unit_id': item['unit_id'],
                'qe_gemba_gpt4_1_mini': None,
                'gemba_gpt4_1_mini_errors': None,
                'gemba_gpt4_1_mini_raw_output': None  
            }
        )
        print (f'Error at row {i}:{e}')

    if i % 100 == 0:
        checkpoint_df_gpt = pd.DataFrame (gpt_results)
        checkpoint_df_gpt.to_csv("../data/dataset_qe/gemba_gpt4_1_mini_qe_outputs.tsv", sep ='\t', index = False)


no minor errors identified.
no minor errors identified.
overall, no critical or major errors; minor issues only.
overall, no critical or major errors; minor issues only.
no major errors
no major errors
overall, the translation is accurate and fluent with only minor stylistic differences.
overall, the translation is accurate and fluent with only minor stylistic differences.
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
no major errors
No error level for the translation accurately conveys the meaning of the source text, is fluent, and uses appropriate terminology and style.
No error level for the translation accurately conveys the meaning of the source text, is fluent, and uses appropriate terminology and style.
no critical or accuracy errors detected; meaning is preserved.
no critical or accuracy errors detected; meaning is preserved.
no major errors
no major

In [19]:
gpt_results_df = pd.DataFrame(gpt_results)

gpt_results_df.to_csv(
    "../data/dataset_qe/gemba_gpt4_1_mini_qe_outputs.tsv",
    sep="\t",
    index=False
)

In [22]:
print(gpt_results_df.shape)
print(gpt_results_df["qe_gemba_gpt4_1_mini"].isna().sum())
gpt_results_df.head(10)


(5160, 4)
0


,unit_id,qe_gemba_gpt4_1_mini,gemba_gpt4_1_mini_errors,gemba_gpt4_1_mini_raw_output
0,flores101-main-tur-100-pe1-1,0,{},No errors detected. \nCategory: no-error
1,flores101-main-tur-100-pe1-2,0,{},No errors detected. The translation accurately...
2,flores101-main-tur-100-pe1-3,-12,{'major': ['accuracy/mistranslation - the turk...,Major:\naccuracy/mistranslation - The Turkish ...
3,flores101-main-tur-100-pe1-4,-7,{'major': ['accuracy/mistranslation - the turk...,Critical:\nno-error\n\nMajor:\naccuracy/mistra...
4,flores101-main-tur-100-pe1-5,-25,"{'critical': ['accuracy/mistranslation - ""bu s...","Critical:\naccuracy/mistranslation - ""bu sabır..."
5,flores101-main-tur-101-pe2-1,-25,{'critical': ['accuracy/omission - the transla...,Critical:\naccuracy/omission - The translation...
6,flores101-main-tur-101-pe2-2,0,{},No errors detected. \nCategory: no-error
7,flores101-main-tur-101-pe2-3,-9,"{'major': ['accuracy/omission - the phrase ""no...",Critical:\nno-error\n\nMajor:\naccuracy/omissi...
8,flores101-main-tur-103-pe1-1,0,{},No errors detected. The translation is accurat...
9,flores101-main-tur-103-pe1-2,-1,{'minor': ['terminology/inappropriate for cont...,Critical:\nno-error\n\nMajor:\nno-error\n\nMin...


In [23]:
# Create final QE dataframe from your existing working dataframe
qe_df = gemba_input_df.copy()

In [24]:
qe_df = qe_df.merge(
    gemma_results_df[
        [
            "unit_id",
            "qe_gemba_gemma3"
        ]
    ],
    on="unit_id",
    how="left",
    validate="one_to_one"
)


In [25]:

qe_df = qe_df.merge(
    gpt_results_df[
        [
            "unit_id",
            "qe_gemba_gpt4_1_mini"
        ]
    ],
    on="unit_id",
    how="left",
    validate="one_to_one"
)

In [30]:
"""
qe_df.to_csv(
    "../data/dataset_qe/divemt_all_qe_scores.tsv",
    sep="\t",
    index=False
)
"""
print(qe_df.shape)
qe_df.head(10)

(5160, 69)


,unit_id,flores_id,item_id,subject_id,task_type,translation_type,src_len_chr,mt_len_chr,tgt_len_chr,src_len_wrd,...,mt_annotations,tgt_tokens,tgt_annotations,src_wmt22_qe,mt_wmt22_qe,qe_cometkiwi,source_lang,target_lang,qe_gemba_gemma3,qe_gemba_gpt4_1_mini
0,flores101-main-tur-100-pe1-1,894,flores101-main-1001,tur_t1,pe1,pe,124,126.0,128,21,...,"[{'lemma': 'yeniden', 'upos': 'ADV', 'feats': ...","['Yeniden', 'giriş', 'şoku', ',', 'kültür', 'ş...","[{'lemma': 'yeniden', 'upos': 'ADV', 'feats': ...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...","['OK', 'OK', 'OK', 'BAD', 'OK', 'OK', 'OK', 'O...",0.863117,English,Turkish,0,0
1,flores101-main-tur-100-pe1-2,895,flores101-main-1002,tur_t1,pe1,pe,137,117.0,117,22,...,"[{'lemma': 'yeni', 'upos': 'ADJ', 'feats': '',...","['Yeni', 'kültüre', 'kolayca', 'uyum', 'sağlay...","[{'lemma': 'yeni', 'upos': 'ADJ', 'feats': '',...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...",0.868266,English,Turkish,0,0
2,flores101-main-tur-100-pe1-3,896,flores101-main-1003,tur_t1,pe1,pe,127,133.0,148,22,...,"[{'lemma': 'yurtdışı', 'upos': 'NOUN', 'feats'...","['Yurtdışında', 'yaşadıktan', 'sonra', 'ülkeni...","[{'lemma': 'yurtdışı', 'upos': 'NOUN', 'feats'...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'BAD', 'B...","['OK', 'OK', 'OK', 'BAD', 'OK', 'BAD', 'OK', '...",0.891622,English,Turkish,0,-12
3,flores101-main-tur-100-pe1-4,897,flores101-main-1004,tur_t1,pe1,pe,134,145.0,166,22,...,"[{'lemma': 'ilk', 'upos': 'ADJ', 'feats': '', ...","['Yurt', 'dışına', 'ilk', 'çıktığınızda', 'çev...","[{'lemma': 'yurt', 'upos': 'NOUN', 'feats': 'C...","['OK', 'OK', 'OK', 'BAD', 'OK', 'BAD', 'BAD', ...","['BAD', 'BAD', 'BAD', 'OK', 'BAD', 'BAD', 'OK'...",0.865835,English,Turkish,-1,-7
4,flores101-main-tur-100-pe1-5,898,flores101-main-1005,tur_t1,pe1,pe,107,89.0,94,15,...,"[{'lemma': 'insan', 'upos': 'NOUN', 'feats': '...","['İnsanlar', ',', 'ülkesine', 'dönenlere', 'de...","[{'lemma': 'insan', 'upos': 'NOUN', 'feats': '...","['OK', 'BAD', 'OK', 'BAD', 'BAD', 'BAD', 'OK',...","['OK', 'BAD', 'BAD', 'BAD', 'BAD', 'OK', 'OK',...",0.635008,English,Turkish,-25,-25
5,flores101-main-tur-101-pe2-1,18,flores101-main-1011,tur_t1,pe2,pe,117,110.0,121,18,...,"[{'lemma': '1960', 'upos': 'PROPN', 'feats': '...","[""1960'larda"", 'Brzezinski', ',', 'John', 'F',...","[{'lemma': '1960', 'upos': 'ADJ', 'feats': 'Ca...","['OK', 'OK', 'OK', 'OK', 'BAD', 'OK', 'OK', 'O...","['OK', 'OK', 'OK', 'BAD', 'BAD', 'OK', 'OK', '...",0.873991,English,Turkish,-1,-25
6,flores101-main-tur-101-pe2-2,19,flores101-main-1012,tur_t1,pe2,pe,157,192.0,192,24,...,"[{'lemma': '1976', 'upos': 'NUM', 'feats': 'Ca...","['1976', 'seçimlerinde', ""Carter'a"", 'dış', 'p...","[{'lemma': '1976', 'upos': 'NUM', 'feats': 'Ca...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...","['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK...",0.895587,English,Turkish,0,0
7,flores101-main-tur-101-pe2-3,20,flores101-main-1013,tur_t1,pe2,pe,275,272.0,285,42,...,"[{'lemma': 'NSA', 'upos': 'NOUN', 'feats': 'Ab...","['NSA', 'olarak', ""Carter'a"", ""1978'deki"", 'Ca...","[{'lemma': 'NSA', 'upos': 'NOUN', 'feats': 'Ab...","['OK', 'OK', 'OK', 'OK', 'BAD', 'OK', 'OK', 'O...","['OK', 'OK', 'OK', 'BAD', 'OK', 'OK', 'BAD', '...",0.870157,English,Turkish,-6,-9
8,flores101-main-tur-103-pe1-1,669,flores101-main-1031,tur_t1,pe1,pe,195,170.0,159,28,...,"[{'lemma': 'bir', 'upos': 'DET', 'feats': 'Def...","['Bir', 'kuruluş', ',', 'eğitim', 'kurumu', 'k...","[{'lemma': 'bir', 'upos': 'DET', 'feats': 'Def...","['BAD', 'OK', 'OK', 'BAD', 'BAD', 'OK', 'BAD',...","['OK', 'BAD', 'BAD', 'BAD', 'BAD', 'BAD', 'BAD...",0.899022,English,Turkish,-1,0
9,flores101-main-tur-103-pe1-2,670,flores101-main-1032,tur_t1,pe1,pe,143,141.0,136,19,...,"[{'lemma': 'bir', 'upos': 'DET', 'feats': 'Def...","['Tüm', 'mevcut', 'kaynaklar', 'bir', 'kuruluş...","[{'lemma': 'tüm', 'upos': 'DET', 'feats': '', ...","['OK', 'BAD', 'BAD', 'BAD', 'OK', 'OK', 'OK', ...","['BAD', 'BAD', '